# Select Positive Examples from 2WikiMultihop Eval Log

Loads the 2WikiMultihop training evaluation log, filters for correct (positive) examples, and displays them with pandas.

Question type field: `type` (values: comparison, compositional, bridge_comparison, inference)

In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

EVAL_LOG = "../logs/twowikimultihop_train_eval.out.0"
TYPE_FIELD = "type"

In [ ]:
def load_eval_log(path):
    with open(path) as f:
        lines = f.readlines()
    samples = []
    for l in lines:
        if 'macro_precision' in l:
            continue
        d = json.loads(l)
        if 'input_sample' in d:
            samples.append(d)
    return samples

samples = load_eval_log(EVAL_LOG)
source = Path(EVAL_LOG).stem
for s in samples:
    s['_source'] = source

print(f"{EVAL_LOG}: {len(samples)} samples")

In [ ]:
positives = [s for s in samples if s['evaluation']['correct'] == 1]
negatives = [s for s in samples if s['evaluation']['correct'] == 0]
dont_know = [s for s in samples if s['evaluation'].get('dont_know') == 1]

print(f"Positive (correct):     {len(positives)}")
print(f"Negative (incorrect):   {len(negatives)}")
print(f"  of which dont_know:   {len(dont_know)}")

In [ ]:
# Show question type distribution
type_counts = Counter(s['input_sample'].get(TYPE_FIELD, 'unknown') for s in positives)
print("Question type distribution (positives):")
for t, c in type_counts.most_common():
    print(f"  {t}: {c}")
print(f"  Total: {sum(type_counts.values())}")

In [ ]:
# Quick peek at one positive sample
positives[0]['input_sample']

In [ ]:
df = pd.DataFrame([{
    'question': s['question'],
    'gt_answer': s['gt_answer'],
    'prediction': s['prediction'],
    'full_prediction': s['full_prediction'],
    'f1': s['evaluation']['f1'],
    'precision': s['evaluation']['precision'],
    'recall': s['evaluation']['recall'],
    'dont_know': s['evaluation']['dont_know'],
    TYPE_FIELD: s['input_sample'].get(TYPE_FIELD),
    'source': s['_source'],
} for s in positives])

print(f"Positive examples: {len(df)}")
df.head(10)

In [ ]:
df[TYPE_FIELD].value_counts()

In [ ]:
# Browse examples by type
for t in df[TYPE_FIELD].unique():
    subset = df[df[TYPE_FIELD] == t]
    print(f"\n{'='*60}")
    print(f"Type: {t} ({len(subset)} samples)")
    print('='*60)
    for _, row in subset.head(3).iterrows():
        print(f"\nQuestion: {row['question']}")
        print(f"Answer: {row['gt_answer']}")
        print(f"Prediction: {row['prediction']}")
        print(f"F1: {row['f1']}")
        print(f"Full:\n{row['full_prediction'][:300]}...")

In [ ]:
# Triples from a sample
positives[0]['triples'][:5]

In [ ]:
# F1 distribution among positives
df['f1'].value_counts().sort_index().plot(kind='bar', title='F1 Distribution (Positive Samples)', figsize=(10, 4));

In [ ]:
# Show a specific sample with full generation
def show_sample(idx):
    s = positives[idx]
    print(f"Question: {s['question']}")
    print(f"Type: {s['input_sample'].get(TYPE_FIELD)}")
    print(f"Ground truth: {s['gt_answer']}")
    print(f"Prediction: {s['prediction']}")
    print(f"F1: {s['evaluation']['f1']}")
    print(f"\nFull generation:\n{s['full_prediction']}")

show_sample(0)

In [ ]:
# Show negatives
neg_df = pd.DataFrame([{
    'question': s['question'],
    'gt_answer': s['gt_answer'],
    'prediction': s['prediction'],
    'f1': s['evaluation']['f1'],
    'dont_know': s['evaluation']['dont_know'],
    TYPE_FIELD: s['input_sample'].get(TYPE_FIELD),
    'source': s['_source'],
} for s in negatives])
print(f"Negative examples: {len(neg_df)}")
neg_df.head(10)

In [ ]:
# Save positives as JSONL
OUTPUT = "../logs/positive_samples_2wiki.jsonl"
with open(OUTPUT, "w") as f:
    for s in positives:
        f.write(json.dumps({
            "question": s["question"],
            "gt_answer": s["gt_answer"],
            "prediction": s["prediction"],
            "full_prediction": s["full_prediction"],
            "prompt": s["prompt"],
            "type": s['input_sample'].get(TYPE_FIELD),
            "evaluation": s["evaluation"],
        }) + "\n")
print(f"Saved {len(positives)} positive samples to {OUTPUT}")